In [8]:
# region Imports
import os
import sys
import numpy as np

from tqdm.notebook import tqdm
sys.path.append("/export/share/peters57dm/Verbund/deepsync/experiments/")

from helper.datasets import (
    load_example,
    load_usps,
    load_htru,
    load_pendigits,
    load_optdigits,
    load_mnist,
    load_letterrecognition,
    load_coil20,
    load_coil100,
    load_har,
    load_mice,
    load_weizmann,
    load_fmnist,
    load_data,
)

sys.path.append("/export/share/peters57dm/Verbund/deepsync/")
from helper.utils import save_dict_as_json, load_json_as_dict

def load_algorithm_predictions(dataname, model_i, algorithm_name):
    results_path = f"/export/share/peters57dm/Verbund/deepsync/results/experiments/competitors302/{algorithm_name}/{dataname}/exp_00/results_{model_i}.json"
    results = load_json_as_dict(results_path)
    return np.array(results["predicted_labels"])

In [9]:
# region Exp. Definition
experiment_params = {
    "competitors": [
        "dipdeck",
        "hdbscan",
        "affinityprop",
        "meanshift"
        ],
    "datasets": [
        load_example,
        load_usps,
        load_htru,
        load_pendigits,
        load_optdigits,
        load_letterrecognition,
        load_har,
        load_mice,
        load_mnist,
        load_fmnist,
        load_coil20,
        load_coil100,
        load_weizmann,
    ],
}
# Hey! that is important too. Don't go too fast my friend :)
execution_params = {
    "experiment_root_path": "/export/share/peters57dm/Verbund/deepsync/results/experiments/N_Clusters",
}
# endregion

In [10]:
# region Params Loading
base_path = execution_params["experiment_root_path"]
algo_names = experiment_params['competitors']
datasets_loading_methods = experiment_params["datasets"]
N_MODELS = 5
# endregion

In [12]:
results = {}
for ds_loader in tqdm(datasets_loading_methods, total=len(datasets_loading_methods)):
    _, _, data_name = load_data(ds_loader)
    results[data_name] = {}
    for algo in algo_names:
        _n_clusters = []
        for model_i in range(N_MODELS):
            preds = load_algorithm_predictions(data_name, model_i, algo)
            n_clusters = len(np.unique(preds))
            _n_clusters.append(n_clusters)

        results[data_name][algo] = f"{np.mean(_n_clusters):.4f}±{np.std(_n_clusters):.4f}"

  0%|          | 0/13 [00:00<?, ?it/s]

In [13]:
save_dict_as_json(
    results, os.path.join(base_path, "n_cluster_results.json")
)

In [14]:
import pandas as pd
pd.DataFrame(results).to_excel(os.path.join(base_path, "n_cluster_results.xlsx"))